In [0]:
# %sql
# DROP TABLE IF EXISTS workspace.formula_1.silver_dim_meetings_sessions;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.formula_1.silver_dim_meetings_sessions (
  session_key INT,
  meeting_key INT,
  season INT,
  meeting_name STRING,
  meeting_official_name STRING,
  location STRING,
  country_name STRING,
  country_code STRING,
  circuit_key INT,
  circuit_short_name STRING,
  circuit_type STRING,
  is_meeting_cancelled BOOLEAN,
  session_name STRING,
  session_type STRING,
  session_start_time TIMESTAMP,
  session_end_time TIMESTAMP,
  gmt_offset STRING
)
USING DELTA;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT meeting_name) AS total_meetings,
    COUNT(DISTINCT location) AS total_locations,
    COUNT(DISTINCT circuit_short_name) AS total_circuit_short_names,
    COUNT(DISTINCT country_code) AS total_country_codes,
    MIN(session_start_time) AS min_date,
    MAX(session_end_time) AS max_date
FROM
    workspace.formula_1.silver_dim_meetings_sessions

In [0]:
%sql
MERGE INTO workspace.formula_1.silver_dim_meetings_sessions AS target
USING (
  WITH parsed_meetings AS (
    SELECT
        TRY_CAST(get_json_object(raw_json, '$.meeting_key') AS INT) AS meeting_key,
        get_json_object(raw_json, '$.meeting_name') AS meeting_name,
        get_json_object(raw_json, '$.meeting_official_name') AS meeting_official_name,
        get_json_object(raw_json, '$.location') AS location,
        get_json_object(raw_json, '$.country_name') AS country_name,
        get_json_object(raw_json, '$.country_code') AS country_code,
        TRY_CAST(get_json_object(raw_json, '$.circuit_key') AS INT) AS circuit_key,
        get_json_object(raw_json, '$.circuit_short_name') AS circuit_short_name,
        get_json_object(raw_json, '$.circuit_type') AS circuit_type,
        TRY_CAST(get_json_object(raw_json, '$.is_cancelled') AS BOOLEAN) AS is_cancelled,
        TRY_CAST(get_json_object(raw_json, '$.year') AS INT) AS year
    FROM
        workspace.formula_1.bronze_meetings
    WHERE
        get_json_object(raw_json, "$.error") IS NULL
  ),
  parsed_sessions AS (
    SELECT
        TRY_CAST(get_json_object(raw_json, '$.session_key') AS INT) AS session_key,
        TRY_CAST(get_json_object(raw_json, '$.meeting_key') AS INT) AS meeting_key,
        get_json_object(raw_json, '$.session_name') AS session_name,
        get_json_object(raw_json, '$.session_type') AS session_type,
        TO_TIMESTAMP(get_json_object(raw_json, '$.date_start')) AS session_start_time,
        TO_TIMESTAMP(get_json_object(raw_json, '$.date_end')) AS session_end_time,
        get_json_object(raw_json, '$.gmt_offset') AS gmt_offset
    FROM
        workspace.formula_1.bronze_sessions
    WHERE
        get_json_object(raw_json, "$.error") IS NULL
  )
  SELECT
    ses.session_key,
    met.meeting_key,
    met.year AS season,
    met.meeting_name,
    met.meeting_official_name,
    met.location,
    met.country_name,
    met.country_code,
    met.circuit_key,
    met.circuit_short_name,
    met.circuit_type,
    met.is_cancelled AS is_meeting_cancelled,
    ses.session_name,
    ses.session_type,
    ses.session_start_time,
    ses.session_end_time,
    ses.gmt_offset
  FROM
    parsed_meetings met
  INNER JOIN
    parsed_sessions ses ON met.meeting_key = ses.meeting_key
) AS source
ON target.session_key = source.session_key
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT meeting_name) AS total_meetings,
    COUNT(DISTINCT location) AS total_locations,
    COUNT(DISTINCT circuit_short_name) AS total_circuit_short_names,
    COUNT(DISTINCT country_code) AS total_country_codes,
    MIN(session_start_time) AS min_date,
    MAX(session_end_time) AS max_date
FROM
    workspace.formula_1.silver_dim_meetings_sessions

In [0]:
%sql
SELECT
*
FROM
workspace.formula_1.silver_dim_meetings_sessions
ORDER BY session_start_time DESC
LIMIT 5